# 🔒 The audit trail that cannot be changed

**Birlasoft FORGE FDE Academy · Sprint 2 Day 33 · DecisionStream AI**

---

Two years from now, somebody will ask why a particular claim was sent for
review. Today you build the record that can answer that question — and that
nobody, including you, can quietly change afterwards.

---

## ⚠️ Read this before you touch Azure

**Use the subscription and the Resource Group that have already been given to
you.** Do not create a new Resource Group. Do not create a new subscription.

Your trainer has set these up so that everything can be cleaned up together at
the end of the programme. A resource sitting in a Resource Group nobody knows
about is a resource that keeps costing money after you have gone home.

**And do not LOCK any immutability policy today.**

This is the one instruction in the whole programme that cannot be undone. A
locked policy cannot be shortened or removed. While data is inside its retention
period you cannot delete the container, and usually you cannot delete the
storage account either. On a shared learner subscription, one person locking a
seven-year policy creates something the whole cohort is stuck with.

We will use an **unlocked policy with one day of retention**. That is enough to
prove every point in this lesson, and it can be cleaned up afterwards.

---

## What is real in this notebook

| Part | Real or simulated |
|---|---|
| The transition record your code writes | **Real** — you write it |
| Losing states in the change feed | **Measured** — you will reproduce Day 18's finding |
| Writing audit records to Blob Storage | **Real** — in your own storage account |
| Trying to delete an audit record and failing | **Real, and this is the point of the day** |
| Trying to overwrite it and failing | **Real** |
| Searching the trail by case_id | **Real** — and you compare two folder layouts |

The delete refusal is the single most useful thing you will produce today. Save
the error message; it goes in your governance appendix.

## What it costs

Almost nothing. A few kilobytes in Standard LRS storage for one day. The
guidance in Appendix A keeps you on the cheapest options that still demonstrate
the lesson properly.

## 0 · Install and import

In [ ]:
%pip install -q azure-storage-blob

import os, re, json, time, uuid, getpass, warnings
from pathlib import Path
from datetime import datetime, timezone, timedelta
from collections import Counter
warnings.filterwarnings("ignore")
print("ready")

## 1 · Set up the storage account in the Azure Portal

Do this in the portal first, then come back. It takes about ten minutes.

### Step 1 — open your existing Resource Group

1. Go to **portal.azure.com** and sign in with the learner account you were given
2. In the top search bar type **Resource groups**
3. **Open the Resource Group that was already created for you.** Do not select
   **+ Create**

> If you cannot see a Resource Group, stop and ask your trainer. Do not create
> one. Creating your own means it will not be cleaned up with everyone else's.

### Step 2 — create a storage account inside that Resource Group

1. Inside your Resource Group, select **+ Create**
2. Search for **Storage account** and select **Create**
3. Fill in the **Basics** tab:

| Field | What to choose | Why |
|---|---|---|
| Subscription | The one already selected | Do not change it |
| **Resource group** | **The existing one** | This is the important line |
| Storage account name | `staudit` + your initials + 3 digits | Must be globally unique, lowercase, no dashes |
| Region | The same region as your Resource Group | Keeps everything together |
| Primary service | Azure Blob Storage | |
| Performance | **Standard** | Premium costs far more and adds nothing here |
| Redundancy | **LRS (Locally-redundant storage)** | **The cheapest option.** GRS copies your data to a second region and roughly doubles the cost |

4. Go to the **Data protection** tab
5. Find **Enable version-level immutability support** and **leave it unticked**

> Leaving that unticked is deliberate. It keeps immutability at the *container*
> level, which is simpler to remove afterwards. Version-level immutability
> enabled at account creation cannot be turned off later.

6. Select **Review + create**, then **Create**. It takes about a minute.

### Step 3 — create the container

1. Open your new storage account
2. In the left menu, under **Data storage**, select **Containers**
3. Select **+ Container**
4. Name it `audit-trail`
5. Leave the access level as **Private**
6. Select **Create**

### Step 4 — put an immutability policy on the container

1. Open the `audit-trail` container
2. At the top select **…** (more) → **Access policy**
3. Under **Immutable blob storage**, select **+ Add policy**
4. Choose **Time-based retention**
5. Set retention to **1 day**
6. Select **Save**

> **Do not select "Lock policy".** An unlocked policy still refuses deletes,
> which is everything you need today. A locked policy cannot be removed, and
> neither can your data, for the whole retention period.

### Step 5 — get the connection string

1. In the storage account, left menu → **Security + networking** → **Access keys**
2. Select **Show** next to **key1**
3. Copy the **Connection string** (not the key on its own — the connection
   string contains the account name and the key together)

> This connection string is a credential. Anyone holding it has full control of
> the storage account. Do not paste it into chat, a document, or a screenshot.
> The cell below hides it as you type.

In [ ]:
CONN = os.environ.get("AZURE_STORAGE_CONNECTION_STRING", "").strip()
if not CONN:
    CONN = getpass.getpass("Paste your storage connection string (hidden): ").strip()

CONTAINER = "audit-trail"

from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import HttpResponseError, ResourceExistsError

svc = BlobServiceClient.from_connection_string(CONN)
container = svc.get_container_client(CONTAINER)

# Confirm what you are actually pointing at BEFORE writing anything.
acct = svc.account_name
props = container.get_container_properties()
print(f"storage account : {acct}")
print(f"container       : {CONTAINER}")
print(f"container exists: yes\n")

# Is the immutability policy actually in place?
imm = props.get("immutable_storage_with_versioning_enabled")
print("Now check in the portal that the container shows a time-based retention")
print("policy of 1 day, in the UNLOCKED state.\n")
print("If you skipped that step, the delete test later will succeed - and a")
print("delete that succeeds proves the opposite of what this lesson is about.")

## 2 · Why your own code must write the record

Before building anything, reproduce the finding from Day 18. This is a local
simulation, and it is faithful to how a change feed behaves.

A change feed is **polled**. It tells you a document changed, and gives you what
that document looks like **now**. Anything that happened between two polls is
already gone.

In [ ]:
def simulate_change_feed(n_cases=50, poll_every_ms=1000, step_ms=120):
    """
    Each case moves through 4 states. The feed is read every poll_every_ms.
    Whatever the document looks like at poll time is what the feed reports.
    """
    STATES = ["received", "extracted", "scored", "routed"]
    clock, docs, transitions = 0, {}, 0
    seen, next_poll = [], poll_every_ms

    for c in range(n_cases):
        cid = f"CLM-2026-{5000+c}"
        for st in STATES:
            docs[cid] = st                  # the document is UPDATED in place
            transitions += 1
            clock += step_ms
            while clock >= next_poll:       # a poll happens here
                seen.append(dict(docs))     # feed sees current state of each doc
                next_poll += poll_every_ms

    # count how many distinct (case, state) pairs the feed ever observed
    observed = set()
    for snap in seen:
        for cid, st in snap.items():
            observed.add((cid, st))
    return transitions, len(observed)

made, saw = simulate_change_feed()
lost = made - saw
print(f"state changes that actually happened : {made}")
print(f"state changes the feed reported      : {saw}")
print(f"LOST                                 : {lost}  ({lost/made:.0%})\n")
print("""No error was raised. No warning appeared. Nothing was retried.

The feed did exactly what it is designed to do. It is a NOTIFIER - it tells you
something changed. It is not a RECORDER.

So if your audit trail depends on the change feed to notice state changes, the
states your system passed through quickly are simply not in your trail. And a
state you never recorded is a state you cannot evidence.""")

## 3 · The transition record

Write it yourself, in the same code path, at the moment the thing happens.

**Append-only** means you only ever add a new row. You never edit an old one and
you never delete one. If something was recorded wrongly, you add a correcting
row — you do not change history.

In [ ]:
AUDIT_SCHEMA = [
  "audit_id", "case_id", "event", "from_state", "to_state",
  "at", "actor", "score", "reasons", "model_version", "prompt_version",
  "correlation_id",
]

def make_audit_record(case_id, event, from_state, to_state, actor,
                      seq=1, **extra):
    """
    audit_id is DETERMINISTIC - same event, same id.

    This matters because your Function may be retried. A random uuid would
    produce two rows for one real event, and an audit trail that double-counts
    is worse than one that is merely incomplete.
    """
    rec = {
      "audit_id": f"{case_id}:{event}:{seq:02d}",
      "case_id": case_id,
      "event": event,
      "from_state": from_state,
      "to_state": to_state,
      "at": datetime.now(timezone.utc).isoformat(timespec="milliseconds"),
      "actor": actor,                      # "system:scoring-fn" or "handler.rb"
      "score": extra.get("score"),
      "reasons": extra.get("reasons", []),
      "model_version": extra.get("model_version", "meridian-anomaly:3"),
      "prompt_version": extra.get("prompt_version", "v4"),
      "correlation_id": extra.get("correlation_id", uuid.uuid4().hex[:12]),
    }
    return rec

# A case moving through its life. Note the ACTOR changes on the last row.
CASE = "CLM-2026-4471"
trail = [
  make_audit_record(CASE, "received",  None,       "received",  "system:intake-fn", 1),
  make_audit_record(CASE, "extracted", "received", "extracted", "system:extract-fn", 1),
  make_audit_record(CASE, "scored",    "extracted","scored",    "system:scoring-fn", 1,
                    score=72.4, reasons=["repeated_bank_account",
                                         "new_policy_high_value"]),
  make_audit_record(CASE, "routed",    "scored",   "priority_review",
                    "system:routing-fn", 1, score=72.4),
  make_audit_record(CASE, "decided",   "priority_review", "settled",
                    "handler.rbramley", 1),
]

print(f"{'event':<12}{'from':<18}{'to':<18}{'actor'}")
print("-"*72)
for r in trail:
    print(f"{r['event']:<12}{str(r['from_state']):<18}{r['to_state']:<18}{r['actor']}")

print("""
LOOK AT THE LAST ROW.

The system routed the case. A NAMED PERSON decided it. Those are two different
events by two different actors, and an auditor can tell them apart.

That separation is the Sprint 0 commitment, visible in the data rather than
asserted in a document. If your trail only says "case completed", you cannot
show whether a system error or a human judgement went wrong.""")

## 4 · Where the files go, and why it matters

Two years from now you have roughly half a million audit records in Blob
Storage and somebody wants one case.

The folder path does the same job a partition key does in Cosmos. It decides
which question is cheap to ask — and unlike a database, **you cannot reorganise
it later, because you are not allowed to delete the files.**

In [ ]:
def safe_name(audit_id):
    """
    Colons are legal in a blob name but they are reserved in a URL, and they
    trip up Storage Explorer, azcopy and anything that builds a path by string
    concatenation. Replace them once, here, rather than debugging it later.
    """
    return audit_id.replace(":", "__")

def path_by_case(rec):
    """Good: one folder per case. Finding one case is one listing."""
    d = rec["at"][:10]
    return f"audit/case_id={rec['case_id']}/{d}/{safe_name(rec['audit_id'])}.json"

def path_by_date(rec):
    """Poor: grouped by date. Finding one case means reading everything."""
    d = rec["at"][:10]
    return f"audit/{d.replace('-','/')}/batch-{safe_name(rec['audit_id'])}.json"

print("LAYOUT A - by case  (what we will use)")
for r in trail[:2]: print("   " + path_by_case(r))
print("\nLAYOUT B - by date  (the natural first guess)")
for r in trail[:2]: print("   " + path_by_date(r))

print("""
To find one case:

  LAYOUT A   list the prefix "audit/case_id=CLM-2026-4471/" and read what is
             there. A handful of reads.

  LAYOUT B   list every folder for every day in range, open every file, check
             whether the case_id inside matches. Tens of thousands of reads,
             and you pay for each one.

Both layouts work. Only one of them answers the question you will actually be
asked - and the question an auditor asks is almost always about ONE CASE.""")

In [ ]:
# Write the trail to Blob Storage.
from azure.storage.blob import ContentSettings

written = []
for rec in trail:
    blob_path = path_by_case(rec)
    blob = container.get_blob_client(blob_path)
    body = json.dumps(rec, indent=2).encode()
    try:
        blob.upload_blob(body, overwrite=False,
                         content_settings=ContentSettings(content_type="application/json"))
        written.append(blob_path)
        print(f"  written  {blob_path}")
    except ResourceExistsError:
        print(f"  exists   {blob_path}  (not overwritten - correct behaviour)")
        written.append(blob_path)

print(f"\n{len(written)} audit records now in the immutable container.")
print("\nNote overwrite=False. An append-only store should refuse a second write")
print("to the same id rather than silently replacing the first one.")

## 5 · The test that matters — try to delete it

This is the point of the day. Saying storage is immutable is a claim. **Showing
a delete being refused is evidence.**

Run this and read the error carefully. Copy the error message somewhere safe —
it goes into your governance appendix.

In [ ]:
target = written[0]
blob = container.get_blob_client(target)

print(f"attempting to DELETE: {target}\n")
try:
    blob.delete_blob()
    print("*** THE DELETE SUCCEEDED ***\n")
    print("That is the wrong result, and it means the immutability policy is not")
    print("actually on this container. Go back to Portal Step 4 and add a")
    print("time-based retention policy of 1 day, UNLOCKED. Then re-run from the")
    print("write cell above.\n")
    print("A delete that succeeds proves the opposite of what this lesson claims.")
except HttpResponseError as e:
    print("DELETE REFUSED - this is the correct result.\n")
    print(f"  status code : {e.status_code}")
    print(f"  error code  : {getattr(e, 'error_code', 'n/a')}")
    msg = str(e).split("\n")[0][:220]
    print(f"  message     : {msg}\n")
    print("""SAVE THIS OUTPUT.

This is the single most useful screenshot in your governance appendix. It is
the difference between telling a regulator the trail is immutable and showing
them that it is.

Note who was refused: you were. Holding the account key, with full control of
the storage account. That is the whole idea - the protection is not a permission
you could grant yourself, it is a property of the container.""")

In [ ]:
# And try to overwrite it, which is the quieter version of the same attack.
print(f"attempting to OVERWRITE: {target}\n")
tampered = json.loads(json.dumps(trail[0]))
tampered["actor"] = "system:definitely-not-tampered"

try:
    container.get_blob_client(target).upload_blob(
        json.dumps(tampered).encode(), overwrite=True)
    print("*** THE OVERWRITE SUCCEEDED - the policy is not protecting you ***")
except HttpResponseError as e:
    print("OVERWRITE REFUSED - correct.\n")
    print(f"  status code : {e.status_code}")
    print(f"  error code  : {getattr(e, 'error_code', 'n/a')}\n")
    print("""Deleting evidence is obvious. Quietly EDITING it is the more realistic
risk - changing who the actor was, or what the score said, months later when
somebody is looking for a reason.

An audit trail you are able to edit is not an audit trail. It is a document.""")

# Prove the original is untouched
back = json.loads(container.get_blob_client(target).download_blob().readall())
print(f"\nactor still recorded as : {back['actor']}")
print(f"unchanged               : {back == trail[0]}")

## 6 · Reading the trail back

An archive you cannot search is a cupboard, not a trail. This is the query an
auditor's question turns into.

In [ ]:
def get_case_trail(case_id):
    """One prefix listing. This is why the folder layout was chosen."""
    prefix = f"audit/case_id={case_id}/"
    rows = []
    for b in container.list_blobs(name_starts_with=prefix):
        rows.append(json.loads(container.get_blob_client(b.name).download_blob().readall()))
    rows.sort(key=lambda r: r["at"])
    return rows

rows = get_case_trail(CASE)
print(f"audit trail for {CASE}   ({len(rows)} events)\n")
print(f"{'time':<26}{'event':<12}{'to state':<18}{'actor'}")
print("-"*82)
for r in rows:
    print(f"{r['at']:<26}{r['event']:<12}{r['to_state']:<18}{r['actor']}")

print(f"""
That is the answer to "show me every step this claim went through".

Every row was written by the code that did the work, at the moment it did it.
None of them can be removed or altered. And finding them took one listing,
because the folder layout was chosen for this question.""")

## 7 · The check to do BEFORE anybody locks a policy

Once a policy is locked, anything personal in these files stays there for the
whole retention period. You cannot delete it, and neither can your client.

This cell is a simple guard. It is not clever, and that is fine — it is meant
to be run before a decision that cannot be reversed.

In [ ]:
# Patterns that suggest personal data has crept into an audit record.
SUSPECT = {
  "email":        r"[\w.+-]+@[\w-]+\.[\w.]+",
  "uk_phone":     r"\b0\d{3}[\s-]?\d{3}[\s-]?\d{3,4}\b",
  "uk_postcode":  r"\b[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}\b",
  "long_number":  r"\b\d{8,}\b",
  "person_name":  r"\b(Mr|Mrs|Ms|Miss|Dr)\s+[A-Z][a-z]+",
}
NAME_LIKE_FIELDS = {"name", "full_name", "claimant", "address", "dob",
                    "date_of_birth", "email", "phone", "account_number"}

def check_record(rec):
    hits = []
    for f in rec:
        if f.lower() in NAME_LIKE_FIELDS:
            hits.append(f"field '{f}' is a personal-data field by name")
    blob = json.dumps(rec)
    for label, pat in SUSPECT.items():
        for m in re.finditer(pat, blob):
            hits.append(f"looks like {label}: {m.group()[:28]}")
    return hits

# Your clean trail, PLUS a deliberately careless record so you can see the
# check actually firing. This is the shape real leakage takes: somebody adds
# "just a bit of context" to make the queue easier to read.
careless = make_audit_record(CASE, "routed", "scored", "priority_review",
                             "system:routing-fn", 1, score=72.4)
careless["claimant"] = "Mrs R Bramley"
careless["reasons"] = ["bank account 20558193 used by 12 claims",
                       "contact r.bramley@example.co.uk for verification"]

print(f"{'audit_id':<46}{'findings'}")
print("-"*104)
total = 0
for r in trail + [careless]:
    h = check_record(r)
    total += len(h)
    label = r["audit_id"] + ("   <-- careless" if r is careless else "")
    print(f"{label:<46}{'; '.join(h) if h else 'none'}")

print(f"""
findings: {total}

Look at the last row. Nobody set out to put personal data in an audit trail.
Somebody added a claimant name and a bit of detail to the reasons field, so the
review queue would be easier to read. That is how it always happens.

If that record had been written to a LOCKED container, the name and the email
address would stay there for the whole retention period - and no request, no
escalation and no amount of budget would remove them.

Run this against a real sample before anybody locks anything. It will not catch
everything - a free-text "reasons" field can carry a name that no pattern
matches - but it catches the obvious cases, and the obvious cases are what
actually happen.

THE THREE WAYS OUT, IF PERSONAL DATA MUST BE REFERENCED

  1  Keep it out.  Store a customer REFERENCE, not a name. Identifiable
     details stay in a normal store you are allowed to delete. Simplest.

  2  Crypto-shredding.  Encrypt each customer's data with its own key. To
     erase them, delete the key. The bytes remain but are unreadable.

  3  A documented legal basis to retain.  A lawyer's decision, in writing,
     before you build. Not yours to make on the day.""")

## 8 · Clean up

The policy is unlocked and the retention is one day, so cleanup is possible.
This is exactly why we did not lock it.

In [ ]:
print("""CLEAN-UP, IN ORDER

1. Wait for the retention period to pass (one day), OR remove the unlocked
   policy first:
      Portal -> your container -> ... -> Access policy
      -> under Immutable blob storage, delete the time-based retention policy

2. Then delete the blobs, or delete the container.

3. Leave the RESOURCE GROUP alone. It is shared, and your trainer will clean
   it up at the end of the programme.

WHAT YOU WOULD NOT BE ABLE TO DO IF YOU HAD LOCKED IT

  - remove the policy                     refused
  - shorten the retention                 refused
  - delete the blobs                      refused until retention expires
  - delete the container                  refused
  - delete the storage account            usually refused

That list is why the instruction at the top of this notebook was to leave it
unlocked. On a shared learner subscription, one locked seven-year policy is
something the whole cohort lives with.""")

# Optional: remove today's blobs if the policy has already been lifted.
REMOVE_NOW = False        # set True only AFTER deleting the policy in the portal
if REMOVE_NOW:
    gone = 0
    for b in container.list_blobs(name_starts_with="audit/"):
        try:
            container.get_blob_client(b.name).delete_blob(); gone += 1
        except HttpResponseError as e:
            print(f"still protected: {b.name} ({getattr(e,'error_code','')})")
    print(f"deleted {gone} blobs")
else:
    print("\nREMOVE_NOW is False. Nothing was deleted.")

---

# Appendix A · Keeping the cost near zero

This is a learning exercise. Every choice below is the cheapest option that
still demonstrates the lesson properly.

| Choice | Pick | Why |
|---|---|---|
| Performance | **Standard** | Premium is priced for high transaction rates you do not have |
| Redundancy | **LRS** | The cheapest. GRS copies to a second region and roughly doubles storage cost |
| Access tier | **Hot** for the lab | Cool storage is cheaper to keep but charges more per read, and you will read a lot today. For a real audit archive, Cool or Archive is the right answer |
| Retention | **1 day** | The minimum. Long enough to prove immutability |
| Policy state | **Unlocked** | A locked policy means you pay until the retention expires, whatever happens |
| Region | Same as your Resource Group | Avoids cross-region data transfer charges |

**What today actually costs:** a handful of kilobytes and a few hundred
operations. It will not be visible on the bill.

**What could cost real money:** locking a long retention policy over a large
container. That is the only expensive mistake available here, and it is
avoidable by following the instruction at the top.

---

# Appendix B · When it does not work

| What you see | What it usually means | What to do |
|---|---|---|
| The delete **succeeds** | No immutability policy on the container | Portal Step 4. Then re-run from the write cell |
| `AuthenticationFailed` | The connection string was copied incompletely | Re-copy the whole string, including `DefaultEndpointsProtocol=` |
| `ContainerNotFound` | Container name typo, or created in a different account | Check the container list in the portal |
| `ResourceExistsError` on write | That audit_id is already stored | Correct. Append-only means one write per id |
| Cannot find **Access policy** | You are looking at the storage account, not the container | Open the container first, then the `…` menu |
| Cannot delete the container afterwards | Retention has not expired, or the policy is locked | Remove the unlocked policy first, or wait |
| `PublicAccessNotPermitted` | The account blocks public access — which is correct | Nothing to fix. The SDK uses the key, not public access |

---

# Questions for the ADR

1. **Who writes your audit record — your code, or something watching from
   outside?** If it is the second, which states are you missing?
2. **Is the record written before the case moves on?** Sprint 0 Day 6 asked for
   this ordering.
3. **What is your folder layout, and which question does it make cheap?** You
   cannot reorganise files you are not allowed to delete.
4. **What personal data is in the record?** Answer before anybody locks
   anything.
5. **What retention period, and who agreed it in writing?**
6. **What does the trail NOT capture?** Every trail has a boundary. Knowing
   yours is the difference between a gap and a surprise.
7. **If the audit Function is down for a day, what happens — and who notices?**

---

### The sentence to carry out of today

> **You cannot prove something you never wrote down.**
> So write it at the moment it happens, in the same code that does the work,
> and put it somewhere that nobody — including you — can quietly change.